In [23]:
import pandas as pd
import numpy as np
import os
import sqlite3
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.linear_model import LogisticRegression
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, classification_report, roc_auc_score
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import shap

os.chdir(r"C:\Users\camde\Desktop\baseball_analytics")

conn = sqlite3.connect("data/baseball.db")

batting = pd.read_sql("SELECT * FROM batting", conn)
batting_post = pd.read_sql("SELECT * FROM batting_post", conn)
allstar = pd.read_sql("SELECT * FROM allstar", conn)
people_raw = pd.read_sql("SELECT * FROM people_raw", conn)
hof = pd.read_sql("SELECT * FROM hall_of_fame", conn)
awards = pd.read_sql("SELECT * FROM awards_players", conn)
series_post = pd.read_sql("SELECT * FROM series_post", conn)
appearances = pd.read_sql("SELECT * FROM appearances", conn)
fielding = pd.read_sql("SELECT * FROM fielding", conn)



In [2]:
for name, df in [("batting", batting), ("people_raw", people_raw), ("batting_post", batting_post), ("allstar", allstar), ("hof", hof), ("awards", awards), ("series_post", series_post)]:
    print(f"\n── {name} ──────────────────────────")
    null_counts = df.isnull().sum().to_frame()
    print(null_counts[null_counts > 0].dropna().astype(int))


── batting ──────────────────────────
          0
lgID    737
RBI     756
SB     2546
CS    32495
SO     8859
IBB   46527
HBP    2816
SH     6068
SF    46664
GIDP  35432

── people_raw ──────────────────────────
                  0
birthYear       978
birthMonth     1352
birthDay       1521
birthCity      1130
birthCountry    939
birthState     1361
deathYear     12237
deathMonth    12244
deathDay      12341
deathCountry  12280
deathState    12352
deathCity     12300
nameFirst       413
weight         2182
height         2008
bats           2847
throws         2140
debut          3030
bbrefID        1275
finalGame      4932
retroID         897

── batting_post ──────────────────────────
        0
CS    734
SO    573
IBB   679
HBP   201
SH    201
SF    917
GIDP  967

── allstar ──────────────────────────
                0
gameNum       689
gameID        738
startingPos  4216

── hof ──────────────────────────
                0
ballots      1789
needed       1946
votes        1983
induc

In [3]:
hof_individual = (hof['inducted'] == 'Y').groupby(hof['playerID']).any().astype(int)
print(hof_individual.head(20))

playerID
aaronha01    1
abbotji01    0
abreubo01    0
adamsba01    0
adamsbo03    0
adamsdo99    0
adamssp01    0
adcocjo01    0
ageeto01     0
aguilri01    0
akerja01     0
alexado01    0
alexape01    1
allenbe01    0
allendi01    1
allenjo02    0
allenne02    0
alleyge01    0
allisdo01    0
alomaro01    1
Name: inducted, dtype: int64


In [4]:
print(series_post.columns.tolist())
series_post = series_post[series_post['round'] == 'WS']
ws_winners = series_post[['yearID', 'teamIDwinner']]
ws_winners.rename(columns={'teamIDwinner': 'teamID'}, inplace=True)
ws_wins_df = pd.merge(ws_winners, appearances, on=['yearID', 'teamID'])
ws_wins_df = ws_wins_df.groupby('playerID')['yearID'].count().reset_index()
ws_wins_df.columns = ['playerID', 'ws_wins']
print(ws_wins_df)


['yearID', 'round', 'teamIDwinner', 'lgIDwinner', 'teamIDloser', 'lgIDloser', 'wins', 'losses', 'ties']
       playerID  ws_wins
0     aaronha01        1
1     abbated01        1
2     abbotgl01        3
3     abbotku01        1
4     abbotpa01        1
...         ...      ...
3232  zitzmbi01        1
3233  zobribe01        2
3234  zoldasa01        1
3235  zoskyed01        1
3236  zuberbi01        1

[3237 rows x 2 columns]


In [5]:
batting_sum_features = batting[['playerID', 'G', 'AB', 'R', 'H', '2B', '3B', 'HR', 'RBI', 'SB', 'CS', 'BB', 'SO', 'IBB', 'HBP', 'SH', 'SF', 'GIDP']]
batting_sum = batting_sum_features.groupby('playerID').sum()
obp_denom = batting_sum['AB'] + batting_sum['BB'] + batting_sum['HBP'] + batting_sum['SF']
obp_denom = obp_denom.replace(0, np.nan)
slg_denom = batting_sum['AB']
slg_denom = slg_denom.replace(0, np.nan)
batting_sum['OBP'] = (batting_sum['H'] + batting_sum['BB'] + batting_sum['HBP'])/(obp_denom)
batting_sum['singles'] = batting_sum['H'] - (batting_sum['2B'] + batting_sum['3B'] + batting_sum['HR'])
batting_sum['SLG'] = (batting_sum['singles'] + 2*batting_sum['2B'] + 3*batting_sum['3B'] + 4*batting_sum['HR'])/slg_denom
batting_sum['OPS'] = batting_sum['SLG'] + batting_sum['OBP']
batting_sum = batting_sum.drop('singles', axis = 1)
print(batting_sum.head(4))

              G     AB     R     H   2B  3B   HR     RBI     SB    CS    BB  \
playerID                                                                      
aardsda01   331      4     0     0    0   0    0     0.0    0.0   0.0     0   
aaronha01  3298  12364  2174  3771  624  98  755  2297.0  240.0  73.0  1402   
aaronto01   437    944   102   216   42   6   13    94.0    9.0   8.0    86   
aasedo01    448      5     0     0    0   0    0     0.0    0.0   0.0     0   

               SO    IBB   HBP    SH     SF   GIDP       OBP       SLG  \
playerID                                                                 
aardsda01     2.0    0.0   0.0   1.0    0.0    0.0  0.000000  0.000000   
aaronha01  1383.0  293.0  32.0  21.0  121.0  328.0  0.373949  0.554513   
aaronto01   145.0    3.0   0.0   9.0    6.0   36.0  0.291506  0.327331   
aasedo01      3.0    0.0   0.0   0.0    0.0    0.0  0.000000  0.000000   

                OPS  
playerID             
aardsda01  0.000000  
aaronha01  0.9

In [6]:
allstar_apps = allstar.groupby('playerID').size().reset_index()
allstar_apps.columns = ['playerID', 'n_apps']

In [7]:
mvp = awards[awards['awardID'] == 'Most Valuable PLayer']
gold_glove = awards[awards['awardID'] == 'Gold Glove']
silver_slugger = awards[awards['awardID'] == 'Silver Slugger']

mvp_count = mvp.groupby('playerID').size().reset_index()
mvp_count.columns = ['playerID', 'mvp']
gg_count = gold_glove.groupby('playerID').size().reset_index()
gg_count.columns = ['playerID', 'gold_glove']
ss_count = silver_slugger.groupby('playerID').size().reset_index()
ss_count.columns = ['playerID', 'silver_slugger']

mvp_gg_awards = pd.merge(mvp_count, gg_count, on='playerID', how='outer')
tot_awards = pd.merge(mvp_gg_awards, ss_count, on='playerID',how = 'outer')
tot_awards.fillna(0, inplace=True)

,playerID,mvp,gold_glove,silver_slugger
0,aaronha01,0.0,3.0,0.0
1,abreubo01,0.0,1.0,1.0
2,abreujo02,0.0,0.0,3.0
3,abreuwi02,0.0,2.0,0.0
4,acunaro01,0.0,0.0,3.0
...,...,...,...,...
643,younger01,0.0,0.0,1.0
644,youngmi02,0.0,1.0,0.0
645,yountro01,0.0,1.0,3.0
646,zambrca01,0.0,0.0,3.0


In [8]:
prim_pos = pd.read_sql("SELECT * FROM v_primary_position", conn)
sum_fielding = fielding.groupby('playerID')[['PO', 'A', 'E']].sum().reset_index()
fielding_num = sum_fielding['PO'] + sum_fielding['A']
fielding_denom =  sum_fielding['PO'] + sum_fielding['A'] + sum_fielding['E']
fielding_denom = fielding_denom.replace(0, np.nan)
sum_fielding['fielding_pct'] = fielding_num/fielding_denom

In [9]:
career_seasons = batting.groupby('playerID')['yearID'].nunique().reset_index()
career_seasons.columns = ['playerID', 'n_seasons']

In [10]:
people_raw.dropna(subset=['finalGame'], inplace = True)
people_raw['age_at_retirement'] = people_raw['finalGame'].str[:4].astype(int) - people_raw['birthYear']
age_at_retirement = people_raw[['playerID', 'age_at_retirement']]

In [11]:
bat_aa = pd.merge(batting_sum, allstar_apps, on='playerID', how='left')
add_tot_awards = pd.merge(bat_aa, tot_awards, on='playerID', how='left')
add_ws_wins = pd.merge(add_tot_awards, ws_wins_df, on='playerID', how='left')
add_prim_pos = pd.merge(add_ws_wins, prim_pos, on='playerID', how='left')
add_fielding = pd.merge(add_prim_pos, sum_fielding, on='playerID', how='left')
add_career_seasons = pd.merge(add_fielding, career_seasons, on='playerID', how='left')
full_df = pd.merge(add_career_seasons, age_at_retirement, on='playerID', how='left').fillna(0)
hof_df = pd.merge(full_df, hof_individual, on='playerID', how='left').fillna(0)

hof_df = hof_df[hof_df['primary_position'] != 'P']

hof_df = hof_df[hof_df['AB'] >= 1000]

In [12]:
hof_df.drop(['PO', 'A', 'E'], axis = 1, inplace=True)
pos_dummies = pd.get_dummies(hof_df['primary_position'], prefix = 'pos').astype(int)
hof_df = pd.concat([hof_df, pos_dummies], axis = 1)
hof_df = hof_df.drop(['primary_position'], axis = 1)
print(hof_df.columns.tolist())

['playerID', 'G', 'AB', 'R', 'H', '2B', '3B', 'HR', 'RBI', 'SB', 'CS', 'BB', 'SO', 'IBB', 'HBP', 'SH', 'SF', 'GIDP', 'OBP', 'SLG', 'OPS', 'n_apps', 'mvp', 'gold_glove', 'silver_slugger', 'ws_wins', 'fielding_pct', 'n_seasons', 'age_at_retirement', 'inducted', 'pos_1B', 'pos_2B', 'pos_3B', 'pos_C', 'pos_OF', 'pos_SS']


In [13]:
print(hof_df.columns.tolist())

['playerID', 'G', 'AB', 'R', 'H', '2B', '3B', 'HR', 'RBI', 'SB', 'CS', 'BB', 'SO', 'IBB', 'HBP', 'SH', 'SF', 'GIDP', 'OBP', 'SLG', 'OPS', 'n_apps', 'mvp', 'gold_glove', 'silver_slugger', 'ws_wins', 'fielding_pct', 'n_seasons', 'age_at_retirement', 'inducted', 'pos_1B', 'pos_2B', 'pos_3B', 'pos_C', 'pos_OF', 'pos_SS']


In [15]:
print(hof_df['inducted'].value_counts())
print(hof_df['inducted'].value_counts(normalize=True))
print(hof_df.shape)

inducted
0.0    3906
1.0     211
Name: count, dtype: int64
inducted
0.0    0.948749
1.0    0.051251
Name: proportion, dtype: float64
(4117, 36)


In [17]:
X = hof_df.drop(['playerID', 'inducted'], axis = 1)
y = hof_df['inducted']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=.2)

In [20]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

LogisticModel = LogisticRegression(class_weight='balanced', max_iter=1000)
LogisticModel.fit(X_train_scaled, y_train) 
y_preds = LogisticModel.predict(X_test_scaled)
log_class_report = classification_report(y_test, y_preds)
log_auc_roc = roc_auc_score(y_test, y_preds)
print(f'class summary: {log_class_report}, auc-roc: {log_auc_roc}')

class summary:               precision    recall  f1-score   support

         0.0       0.99      0.92      0.96       785
         1.0       0.35      0.87      0.50        39

    accuracy                           0.92       824
   macro avg       0.67      0.90      0.73       824
weighted avg       0.96      0.92      0.93       824
, auc-roc: 0.8964069900375633


In [ ]:
scale_pos_weight = (y_train == 0).sum()/ (y_train == 1).sum()

18.1453488372093


In [22]:
os.makedirs('models', exist_ok=True)

param_grid = {
    'n_estimators': [100, 200],
    'learning_rate': [.01, .1],
    'max_depth': [3, 5],
    'scale_pos_weight': [scale_pos_weight]
}

if os.path.exists('models/xgb_class_grid_search.pkl'):
    xgb_grid_search = joblib.load('models/xgb_class_grid_search.pkl')
else:
    xgb_grid_search = GridSearchCV(
        estimator=XGBClassifier(),
        param_grid=param_grid,
        cv=5,
        scoring='roc_auc'
    )
    xgb_grid_search.fit(X_train, y_train)
    joblib.dump(xgb_grid_search, 'models/xgb_class_grid_search.pkl')

y_preds = xgb_grid_search.best_estimator_.predict(X_test)

xgb_class_report = classification_report(y_test, y_preds)
xgb_auc_roc = roc_auc_score(y_test, y_preds)
print(f'class summary: {xgb_class_report}, auc-roc: {xgb_auc_roc}')

class summary:               precision    recall  f1-score   support

         0.0       0.99      0.97      0.98       785
         1.0       0.59      0.74      0.66        39

    accuracy                           0.96       824
   macro avg       0.79      0.86      0.82       824
weighted avg       0.97      0.96      0.97       824
, auc-roc: 0.8590560182916871


In [29]:
X_train_scaled_df = pd.DataFrame(X_train_scaled, columns=X_train.columns)
X_test_scaled_df = pd.DataFrame(X_test_scaled, columns=X_test.columns)

shap_explainer = shap.LinearExplainer(LogisticModel, X_train_scaled_df)
shap_values = shap_explainer.shap_values(X_test_scaled_df)
shap.summary_plot(shap_values, X_test_scaled_df, plot_type = 'bar', show = False)
plt.savefig('outputs/figures/shap_hof.png', dpi=150, bbox_inches='tight')
plt.close()

In [31]:
def make_hof(playerID):
    if playerID in hof_df['playerID'].values:
        player_data = hof_df[hof_df['playerID'] == playerID]
        player_data.drop(['playerID','inducted'], axis = 1, inplace=True)
        player_data_scaled = scaler.transform(player_data)
        hof_pred = LogisticModel.predict_proba(player_data_scaled)[0][1]
        return f'HOF Chance: {hof_pred}'
    else:
        return f'Sorry that playerID does not exist'
    
make_hof('abreubo01')

'HOF Chance: 0.24845934562960773'